In [2]:
%%capture
pip install xgboost catboost

In [3]:
import warnings, math
warnings.filterwarnings("ignore")
import calendar

import numpy as np
import pandas as pd
from pandas.api.types import is_datetime64_any_dtype, is_object_dtype, is_string_dtype
import matplotlib.pyplot as plt

from datetime import timedelta

# Métricas y visuales
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
    confusion_matrix, precision_score, recall_score,
    brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import confusion_matrix

# Opcionales (se activan si están instalados)
try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

try:
    from catboost import CatBoostClassifier, Pool
    HAS_CAT = True
except Exception:
    HAS_CAT = False
    

    
ID_COLS = {
    "user_id", "userid",
    "channelUserIdentifier",
    "premia_accountid", "accountid", "member_id",
    "spin_user_id", "id"
}


In [4]:
from google.cloud import bigquery
client = bigquery.Client(project="spin-aip-singularity-comp-sb")

query = """
SELECT * 
FROM `spin-aip-singularity-comp-sb.model_activation.dataste_model_activation_timewindow_reduction_V-1-4-1` 
"""
data = client.query(query).to_dataframe()

In [29]:


# =========================================================
# 0) Helpers de thresholds (rápidos)
# =========================================================
def best_threshold_by_cost(y_true, y_proba, cost_fp=1.0, cost_fn=1.0, max_points=200):
    """
    Busca el umbral que minimiza costo esperado, probando pocos puntos.
    """
    y_true = np.array(y_true)
    y_proba = np.array(y_proba)
    qs = np.linspace(0, 1, max_points)
    thrs = np.quantile(y_proba, qs)

    best_thr = 0.5
    best_cost = 1e18

    for t in thrs:
        y_hat = (y_proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0,1]).ravel()
        cost = fp * cost_fp + fn * cost_fn
        if cost < best_cost:
            best_cost = cost
            best_thr = float(t)
    return best_thr


def fast_best_threshold_max_npv(y_true, y_proba, max_points=200, sample_size=200_000):
    """
    Versión rápida para datasets enormes.
    """
    y_true = np.array(y_true)
    y_proba = np.array(y_proba)
    n = len(y_true)
    if n > sample_size:
        idx = np.random.choice(n, size=sample_size, replace=False)
        y_true_s = y_true[idx]
        y_proba_s = y_proba[idx]
    else:
        y_true_s = y_true
        y_proba_s = y_proba

    qs = np.linspace(0, 1, max_points)
    thrs = np.quantile(y_proba_s, qs)

    best_thr = 0.5
    best_npv = -1.0

    for t in thrs:
        y_hat = (y_proba_s >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true_s, y_hat, labels=[0,1]).ravel()
        denom = (tn + fn)
        npv = tn / denom if denom > 0 else 0.0
        if npv > best_npv:
            best_npv = npv
            best_thr = float(t)
    return best_thr

In [30]:
# =========================================================
# 1) train_eval_horizon parcheado
#     - usa tus funciones del notebook:
#       get_xy_for_horizon, onehot_preprocessor, make_histgb
# =========================================================
def train_eval_horizon(hname,
                       label_col,
                       gap_before_days=0,
                       embargo_after_days=0,
                       cost_fp=1.0,
                       cost_fn=1.0,
                       calib_method="isotonic"):
    """
    Entrena un HistGB sobre el horizonte dado y regresa métricas OOF-like.
    Asume que get_xy_for_horizon(hname, label_col) ya implementa la censura temporal.
    """
    # 1) obtener datos
    X, y, feats, num_cols, cat_cols, k = get_xy_for_horizon(hname, label_col)

    # 2) prepro + modelo
    pre_dense = onehot_preprocessor(
        num_cols,
        cat_cols,
        scale_numeric=False,
        min_freq=0.01,
        dense_ohe=True
    )
    model = make_histgb(pre_dense)

    # 3) fit full (si quieres KFold aquí, lo metes, pero tu notebook ya hace folds arriba)
    model.fit(X[feats], y)

    # 4) predicciones
    y_prob = model.predict_proba(X[feats])[:, 1]

    # 5) métricas globales
    try:
        ap = average_precision_score(y, y_prob)
    except ValueError:
        ap = None
    try:
        auc = roc_auc_score(y, y_prob)
    except ValueError:
        auc = None

    # 6) thresholds
    thr_cost = best_threshold_by_cost(y, y_prob, cost_fp=cost_fp, cost_fn=cost_fn, max_points=150)
    thr_npv  = fast_best_threshold_max_npv(y, y_prob, max_points=150, sample_size=150_000)

    # 7) eval @ thr_cost
    y_hat_cost = (y_prob >= thr_cost).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, y_hat_cost, labels=[0,1]).ravel()
    npv_cost = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    prec_cost = precision_score(y, y_hat_cost, zero_division=0)
    rec_cost = recall_score(y, y_hat_cost, zero_division=0)

    eval_cost = {
        "npv": npv_cost,
        "precision": prec_cost,
        "recall": rec_cost
    }

    # 8) eval @ thr_npv
    y_hat_npv = (y_prob >= thr_npv).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, y_hat_npv, labels=[0,1]).ravel()
    npv_npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    prec_npv = precision_score(y, y_hat_npv, zero_division=0)
    rec_npv = recall_score(y, y_hat_npv, zero_division=0)

    eval_npv = {
        "npv": npv_npv,
        "precision": prec_npv,
        "recall": rec_npv
    }

    # 9) armar resultado
    res = {
        "HistGB": {
            "model": model,          # lo guardamos aquí
            "oof_ap": ap,
            "oof_auc": auc,
            "thr_cost": thr_cost,
            "thr_npv": thr_npv,
            "eval_cost": eval_cost,
            "eval_npv": eval_npv
        }
    }
    return res


In [31]:
# =========================================================
# 2) Entrenar por horizontes y guardar en ALL_RESULTS
# =========================================================
if "ALL_RESULTS" not in globals():
    ALL_RESULTS = {}

HORIZONS = [
    ("W0_discrete",  "y_w0",  0,  1),  # primeras horas
    ("W1_discrete",  "y_w1",  0,  1),  # resto del día
    ("W7_discrete",  "y_w7",  1,  7),  # hasta 7 días
    ("W30_discrete", "y_w30", 7, 30),  # hasta 30 días
]

for hname, lcol, gap_days, embargo_days in HORIZONS:
    # si ya existe, no lo vuelvas a correr
    if hname in ALL_RESULTS and "HistGB" in ALL_RESULTS[hname] and "model" in ALL_RESULTS[hname]["HistGB"]:
        continue

    res = train_eval_horizon(
        hname,
        lcol,
        gap_before_days=gap_days,
        embargo_after_days=embargo_days,
        cost_fp=2.0,
        cost_fn=2.0,
        calib_method="isotonic"
    )
    ALL_RESULTS[hname] = res

In [32]:
# =========================================================
# 3) Helper para sacar probas con el modelo ya guardado
# =========================================================
def get_oof_scores(hname, label_col):
    X, y, feats, num_cols, cat_cols, k = get_xy_for_horizon(hname, label_col)
    model = ALL_RESULTS[hname]["HistGB"]["model"]
    proba = model.predict_proba(X[feats])[:, 1]
    return X.index, proba

In [33]:
# =========================================================
# 4) Scores por horizonte → tabla de scoring
# =========================================================
idx_w0,  p_w0  = get_oof_scores("W0_discrete",  "y_w0")
idx_w1,  p_w1  = get_oof_scores("W1_discrete",  "y_w1")
idx_w7,  p_w7  = get_oof_scores("W7_discrete",  "y_w7")
idx_w30, p_w30 = get_oof_scores("W30_discrete", "y_w30")

df_scores = df_lab[["user_id", "signup_date", "channelDetail"]].copy()

df_scores.loc[idx_w0,  "p_w0"]  = p_w0
df_scores.loc[idx_w1,  "p_w1"]  = p_w1
df_scores.loc[idx_w7,  "p_w7"]  = p_w7
df_scores.loc[idx_w30, "p_w30"] = p_w30

In [34]:
# =========================================================
# 5) Derivar score_dia / score_semana / score_mes
# =========================================================
df_scores["score_dia"] = (
    df_scores["p_w0"].fillna(0.0) +
    df_scores["p_w1"].fillna(0.0)
).clip(0, 1)

df_scores["score_semana"] = (
    df_scores["score_dia"] +
    df_scores["p_w7"].fillna(0.0)
).clip(0, 1)

df_scores["score_mes"] = (
    df_scores["score_semana"] +
    df_scores["p_w30"].fillna(0.0)
).clip(0, 1)

In [35]:
# =========================================================
# 6) Segmentación de prob de conocer su primera tx
# =========================================================
def bucket_prob(p):
    if p >= 0.70:
        return "Alta"
    elif p >= 0.40:
        return "Media"
    else:
        return "Baja"

df_scores["prob_first_tx_dia"]    = df_scores["score_dia"]
df_scores["prob_first_tx_semana"] = df_scores["score_semana"]
df_scores["prob_first_tx_mes"]    = df_scores["score_mes"]

df_scores["seg_first_tx_dia"]    = df_scores["prob_first_tx_dia"].apply(bucket_prob)
df_scores["seg_first_tx_semana"] = df_scores["prob_first_tx_semana"].apply(bucket_prob)
df_scores["seg_first_tx_mes"]    = df_scores["prob_first_tx_mes"].apply(bucket_prob)


In [36]:
# =========================================================
# 7) Canal de transacción más viable por ventana
#    (depende de que df_lab tenga first_tx_type o activation_channel)
# =========================================================
TX_COL = "first_tx_type" if "first_tx_type" in df_lab.columns else (
    "activation_channel" if "activation_channel" in df_lab.columns else None
)

if TX_COL is not None:
    # máscaras por ventana real
    m_dia = (df_lab["y_w0"] == 1) | (df_lab["y_w1"] == 1)
    m_sem = m_dia | (df_lab["y_w7"] == 1)
    m_mes = m_sem | (df_lab["y_w30"] == 1)

    canal_dia = (
        df_lab.loc[m_dia]
        .groupby("channelDetail")[TX_COL]
        .agg(lambda s: s.value_counts().index[0] if s.notna().any() else None)
        .rename("canal_tx_mas_viable_dia")
    )

    canal_semana = (
        df_lab.loc[m_sem]
        .groupby("channelDetail")[TX_COL]
        .agg(lambda s: s.value_counts().index[0] if s.notna().any() else None)
        .rename("canal_tx_mas_viable_semana")
    )

    canal_mes = (
        df_lab.loc[m_mes]
        .groupby("channelDetail")[TX_COL]
        .agg(lambda s: s.value_counts().index[0] if s.notna().any() else None)
        .rename("canal_tx_mas_viable_mes")
    )

    df_scores = (
        df_scores
        .merge(canal_dia,    on="channelDetail", how="left")
        .merge(canal_semana, on="channelDetail", how="left")
        .merge(canal_mes,    on="channelDetail", how="left")
    )

In [37]:
# =========================================================
# 8) Resumen de resultados
# =========================================================
def summarize_results(allres):
    rows = []
    for hname, d in allres.items():
        for m, r in d.items():
            rows.append({
                "horizon": hname,
                "model": m,
                "AUC-PR": r.get("oof_ap"),
                "AUC-ROC": r.get("oof_auc"),
                "thr_cost": r.get("thr_cost"),
                "thr_npv": r.get("thr_npv"),
                "NPV@cost": r.get("eval_cost", {}).get("npv"),
                "Prec@cost": r.get("eval_cost", {}).get("precision"),
                "Rec@cost": r.get("eval_cost", {}).get("recall"),
                "NPV@npv":  r.get("eval_npv", {}).get("npv"),
                "Prec@npv": r.get("eval_npv", {}).get("precision"),
                "Rec@npv":  r.get("eval_npv", {}).get("recall"),
            })
    return pd.DataFrame(rows).sort_values(["horizon","AUC-PR"], ascending=[True,False])

In [38]:
summary_df = summarize_results(ALL_RESULTS)

In [39]:
display(summary_df.head(100))

,horizon,model,AUC-PR,AUC-ROC,thr_cost,thr_npv,NPV@cost,Prec@cost,Rec@cost,NPV@npv,Prec@npv,Rec@npv
0,W0_discrete,HistGB,0.444465,0.686978,0.486754,0.064809,0.723398,0.545994,0.088679,0.963937,0.290924,0.999185
1,W1_discrete,HistGB,0.154512,0.684637,0.377703,0.010083,0.914910,1.000000,0.000005,0.994380,0.085630,0.999554
3,W30_discrete,HistGB,0.204257,0.694675,0.556624,0.000067,0.902214,0.000000,0.000000,1.000000,0.098448,1.000000
2,W7_discrete,HistGB,0.266170,0.698338,0.562747,0.015270,0.861364,0.000000,0.000000,0.998797,0.139584,0.999941


In [40]:
df_scores.head()

,user_id,signup_date,channelDetail,p_w0,p_w1,p_w7,p_w30,score_dia,score_semana,score_mes,prob_first_tx_dia,prob_first_tx_semana,prob_first_tx_mes,seg_first_tx_dia,seg_first_tx_semana,seg_first_tx_mes,canal_tx_mas_viable_dia,canal_tx_mas_viable_semana,canal_tx_mas_viable_mes
0,cf827f9c-1854-410c-9cf7-a5fba8dbdbaa,2025-08-11,POS,0.372796,0.092569,0.325048,0.119999,0.465365,0.790413,0.910412,0.465365,0.790413,0.910412,Media,Alta,Alta,CASH_IN_AT_OXXO,CASH_IN_AT_OXXO,CASH_IN_AT_OXXO
1,cf8479e7-481a-43f1-9300-377c13f9d58d,2025-10-05,POS,0.376468,0.123579,0.151844,0.071992,0.500047,0.651890,0.723882,0.500047,0.651890,0.723882,Media,Media,Alta,CASH_IN_AT_OXXO,CASH_IN_AT_OXXO,CASH_IN_AT_OXXO
2,cf8b3659-a1e3-453f-8fa8-36c8e1d1c924,2025-02-04,POS,0.373329,0.159688,0.170688,0.086614,0.533016,0.703705,0.790318,0.533016,0.703705,0.790318,Media,Alta,Alta,CASH_IN_AT_OXXO,CASH_IN_AT_OXXO,CASH_IN_AT_OXXO
3,cf8ca45e-319c-4b7b-8e78-be28d19d0b43,2025-07-09,POS,0.367602,0.160582,0.195781,0.102410,0.528184,0.723965,0.826375,0.528184,0.723965,0.826375,Media,Alta,Alta,CASH_IN_AT_OXXO,CASH_IN_AT_OXXO,CASH_IN_AT_OXXO
4,cf954b0e-1e9f-41b3-9c0a-4c1d2d7d1ed9,2025-06-10,POS,0.373306,0.080335,0.164029,0.109757,0.453641,0.617670,0.727427,0.453641,0.617670,0.727427,Media,Media,Alta,CASH_IN_AT_OXXO,CASH_IN_AT_OXXO,CASH_IN_AT_OXXO


In [44]:
df_scores.describe().T

,count,mean,std,min,25%,50%,75%,max
p_w0,2426355.0,0.289239,0.129507,0.033059,0.175551,0.292588,0.398265,0.749165
p_w1,2426355.0,0.085100,0.048868,0.004648,0.042636,0.083385,0.116739,0.377703
p_w7,2426355.0,0.138614,0.083844,0.001357,0.073378,0.132633,0.177592,0.562747
p_w30,2426355.0,0.097814,0.062957,0.000042,0.057243,0.088489,0.118305,0.556624
score_dia,2426355.0,0.374338,0.159575,0.048190,0.238989,0.387845,0.516140,0.784380
score_semana,2426355.0,0.512889,0.221421,0.062949,0.321523,0.555458,0.695332,1.000000
score_mes,2426355.0,0.608923,0.244589,0.063016,0.391814,0.677437,0.802228,1.000000
prob_first_tx_dia,2426355.0,0.374338,0.159575,0.048190,0.238989,0.387845,0.516140,0.784380
prob_first_tx_semana,2426355.0,0.512889,0.221421,0.062949,0.321523,0.555458,0.695332,1.000000
prob_first_tx_mes,2426355.0,0.608923,0.244589,0.063016,0.391814,0.677437,0.802228,1.000000
